# Exam RAG: fine-tune the question extractor on Colab

LoRA fine-tune of **Qwen2.5-3B-Instruct** on `data_v3/`. This does on an NVIDIA GPU what `finetune.sh` does with MLX on a Mac,
with the same settings: LoRA, learning rate 1e-4, about 2 epochs, up to 4,096 tokens, and loss on the answer only (`--mask-prompt`).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free tier is enough).
2. Upload `colab_bundle.zip` to **My Drive / exam_rag_colab /**. It holds `data_v3/`, `sample_pdfs/` and the evaluation scripts.
   To rebuild it on your Mac, from the project folder:
   `zip -r colab_bundle.zip data_v3 sample_pdfs evaluate.py prep_utils.py prompts.py cleaning.py`
3. *Runtime → Run all*. Checkpoints go to Google Drive every 50 steps. If Colab disconnects, reconnect and **Run all** again: training resumes from the last checkpoint.

**What you get** (in My Drive / exam_rag_colab /)
- `adapter/`: the LoRA weights (small)
- `exam_rag_model_v2/`: the merged model (~6 GB), the same format as your current `exam_rag_model/`
- `evaluation_v2.jsonl`: scores on the labelled sample papers, to compare with the current model

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# transformers and torch come with Colab; add the LoRA / 4-bit / PDF pieces
!pip install -q -U peft bitsandbytes accelerate pdfplumber

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile

DRIVE_DIR = "/content/drive/MyDrive/exam_rag_colab"
WORK_DIR = "/content/exam_rag"
bundle = f"{DRIVE_DIR}/colab_bundle.zip"
assert os.path.exists(bundle), f"Upload colab_bundle.zip to {DRIVE_DIR} first"
with zipfile.ZipFile(bundle) as z:
    z.extractall(WORK_DIR)
print(sorted(os.listdir(WORK_DIR)))

## Settings
Same as `finetune.sh`. On a T4, batch size 1 with 8 gradient-accumulation steps gives an effective batch of 8.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"     # the base of exam_rag_model and finetune.sh
OUTPUT_DIR = f"{DRIVE_DIR}/checkpoints"      # on Drive, so a disconnect doesn't lose the run
ADAPTER_DIR = f"{DRIVE_DIR}/adapter"
MERGED_DIR = f"{DRIVE_DIR}/exam_rag_model_v2"

MAX_LEN = 4096          # longer examples are skipped, never cut (a cut answer teaches stopping early)
LEARNING_RATE = 1e-4
EPOCHS = 2
GRAD_ACCUM = 8
LORA_R = 16
SAVE_EVERY = 50         # steps between checkpoints / validation-loss checks

## Data
Each example is system prompt + PDF text → JSON answer. Only the answer tokens count toward the loss,
so the model learns to produce the JSON rather than to predict the PDF text it was given.

In [ ]:
import json
import torch
import transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def encode(example):
    messages = example["messages"]
    # Render to text, then tokenize: works the same across transformers versions
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    assert full.startswith(prompt), "chat template changed: the prompt is not a prefix of the full chat"
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(full[len(prompt):], add_special_tokens=False)["input_ids"]   # answer + <|im_end|>
    return {"input_ids": prompt_ids + answer_ids, "labels": [-100] * len(prompt_ids) + answer_ids}


def load(path):
    rows, skipped = [], 0
    with open(path) as f:
        for line in f:
            row = encode(json.loads(line))
            if len(row["input_ids"]) > MAX_LEN:
                skipped += 1
                continue
            rows.append(row)
    return rows, skipped


train_rows, train_skipped = load(f"{WORK_DIR}/data_v3/train.jsonl")
val_rows, val_skipped = load(f"{WORK_DIR}/data_v3/valid.jsonl")
lengths = sorted(len(r["input_ids"]) for r in train_rows)
print(f"train {len(train_rows)} (skipped {train_skipped} over {MAX_LEN} tokens) | val {len(val_rows)} (skipped {val_skipped})")
print(f"tokens per example: median {lengths[len(lengths) // 2]}, max {lengths[-1]}")


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), width), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), width), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attention_mask[i, :n] = 1
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

## Model: 4-bit base + LoRA (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# transformers 4.56 renamed torch_dtype to dtype
DTYPE_KW = "dtype" if tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 56) else "torch_dtype"
USE_BF16 = torch.cuda.is_bf16_supported()     # False on a T4 -> fp16
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True),
    device_map={"": 0},
    **{DTYPE_KW: COMPUTE_DTYPE},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

## Train
Validation loss is printed every 50 steps; the checkpoint with the lowest one is kept at the end.
Re-running this cell after a disconnect resumes from the last checkpoint on Drive.

In [ ]:
import os
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=SAVE_EVERY,
    save_strategy="steps",
    save_steps=SAVE_EVERY,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
)
trainer = Trainer(model=model, args=args, data_collator=collate,
                  train_dataset=ListDataset(train_rows), eval_dataset=ListDataset(val_rows))

last = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
print(f"Resuming from {last}" if last else "Starting fresh")
trainer.train(resume_from_checkpoint=last)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

## Quick look: one validation paper

In [ ]:
model = trainer.model
model.eval()
example = json.loads(open(f"{WORK_DIR}/data_v3/valid.jsonl").readline())
prompt = tokenizer.apply_chat_template(example["messages"][:-1], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False, use_cache=True)
print("MODEL:\n", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)[:1500])
print("\nEXPECTED:\n", example["messages"][-1]["content"][:1500])

## Merge the adapter into a full model
Produces the same kind of folder as `exam_rag_model/` (~6 GB on Drive), which `evaluate.py` and MLX load directly.

In [ ]:
import gc
from peft import PeftModel

for name in ("trainer", "model"):
    globals().pop(name, None)       # free the 4-bit training copy
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map={"": 0}, **{DTYPE_KW: torch.float16})
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

del base, merged
gc.collect()
torch.cuda.empty_cache()

## Evaluate on the labelled sample papers
The same test as on your Mac. Current model (`exam_rag_model`) on the 5 held-out papers: **81% of questions found,
100% grounded (none invented), 82% marks correct, year 80% / semester 100%**. Takes ~15–30 minutes on a T4.

In [ ]:
%cd {WORK_DIR}
!python evaluate.py --backend hf --model "{MERGED_DIR}" --output "{DRIVE_DIR}/evaluation_v2.jsonl"

## Back on your Mac
1. Download `exam_rag_colab/exam_rag_model_v2/` from Google Drive into the project folder.
2. Compare on the same papers with the faster MLX backend:
   `python evaluate.py --model exam_rag_model_v2`
3. If it beats the current model, use it for extracting questions from the new papers.